In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
target_df=spark.table('voltgrid.silver.payments')

In [0]:
df=spark.table('voltgrid.bronze.payments')

In [0]:
df=df.withColumn('rnk',F.rank().over(Window.partitionBy('payment_id').orderBy(F.desc('ingestion_time')))).filter(F.col('rnk')==1).drop('rnk','ingestion_time','source_path')

In [0]:
source_df=df.withColumn("gateway",F.upper(F.trim(F.col("gateway"))))\
    .withColumn("amount_aud", F.col("amount_aud").cast("decimal(18,4)"))\
        .withColumn("gst",F.col("gst").cast("decimal(18,4)"))\
            .withColumn("payment_mode",F.upper(F.trim(F.col("payment_mode"))))\
                .withColumn("status",F.upper(F.trim(F.col("status"))))

### `SCD Type 2` Process Start

###  Get Change Records

In [0]:
history_records_df=target_df.filter(F.col("active_flg")=="N")

In [0]:
hash_keys = ['payment_id','customer_id','gateway','amount_aud','gst','payment_mode','processed_at','status']
target_df=target_df.withColumn('hash_key',F.xxhash64(*[F.trim(F.col(c).cast("string")) for c in hash_keys]))
source_df=source_df.withColumn('hash_key',F.xxhash64(*[F.trim(F.col(c).cast("string")) for c in hash_keys]))

In [0]:
active_target_df=target_df.filter(F.col('active_flg')=='Y')

## `changes_records` records which are modified(avilable in target but modifed in source)

In [0]:
changes_records=active_target_df.alias("tar").join(source_df.alias("src"), F.col("tar.payment_id")==F.col("src.payment_id"),'inner').filter(F.col("tar.hash_key") != F.col("src.hash_key")).select("tar.*").withColumn('active_flg',F.lit('N')).withColumn('end_date',F.current_date()).drop("hash_key")

In [0]:
active_target_df=active_target_df.drop("hash_key")
source_df=source_df.drop("hash_key")

## `old_value` old records records which have no update

In [0]:
old_value=active_target_df.alias("t").join(changes_records.alias("c"),F.col("t.payment_id")==F.col("c.payment_id"),how='left_anti')

## `new_records` records needs to insert

In [0]:
new_records=source_df.alias("s").join(active_target_df.alias("t"),F.col("s.payment_id")==F.col("t.payment_id"),how='left_anti').withColumn('active_flg',F.lit('Y')).withColumn('start_date',F.current_date()).withColumn('end_date',F.lit(None).cast('date'))

## `change record values` target records which are changed in source

In [0]:
change_record_values=source_df.alias('src').join(changes_records.alias('cr'), F.col("src.payment_id")==F.col("cr.payment_id"),'left_semi')\
    .withColumn('active_flg',F.lit('Y'))\
        .withColumn('start_date',F.current_date())\
            .withColumn('end_date',F.lit(None).cast('date'))

In [0]:
total_records_df=(history_records_df
        .unionByName(changes_records)
            .unionByName(old_value)
                .unionByName(new_records)
                    .unionByName(change_record_values))

In [0]:
total_records_df.write.format('delta').mode('overwrite').option("overwriteSchema","true").saveAsTable("voltgrid.silver.payments")